# F1-M03: Reportes Sweetviz — Datos Limpios

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 ¿Qué hace este notebook?

Genera **reportes Sweetviz** de los datos **DESPUÉS** de limpiar.
Estos reportes permiten comparar cómo han quedado los datos tras la limpieza
(M02) frente a los datos crudos (M01).

La comparación es clave para verificar que:
- Los renombres de columnas son correctos
- Los valores vacíos se han convertido a NaN
- Los tipos de datos son los esperados
- No se han perdido registros en el proceso

## ⚠️ Requisitos

- Haber ejecutado `f1_m02_limpieza.ipynb` (genera los parquets limpios)
- Los 9 parquets limpios en `data/01_interim/`
- La librería `sweetviz` instalada

## 📦 ¿Qué genera?

| Archivo | Ubicación | Descripción |
|---|---|---|
| 9 reportes Sweetviz | `docs/html/fase1/reportes_clean/` | Uno por cada tabla limpia |
| m03_reportes_clean.html | `docs/html/fase1/` | Página índice con KPIs y enlaces |

## 📁 Flujo de datos

```
data/01_interim/*.parquet  →  9 reportes Sweetviz (datos limpios)
```

## 📌 Siguiente

- `f1_m04a_union_tablas.ipynb` (unión de tablas en df_alumno)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Esta celda hace 3 cosas:
#   1. Importa las librerías necesarias (pandas, numpy, sweetviz)
#   2. Detecta el entorno (Colab o Local) y encuentra la raíz del proyecto
#   3. Importa las rutas y funciones de src.config
#
# El fix de numpy es necesario porque Sweetviz usa internamente
# np.VisibleDeprecationWarning que en numpy >=2.0 se movió.
# ============================================================================

import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

# Silenciar warnings (Sweetviz genera muchos)
warnings.filterwarnings('ignore')

# --- Fix para numpy + Sweetviz ---
if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning

# --- Importar Sweetviz ---
import sweetviz as sv
print(f'✅ Sweetviz {sv.__version__}')

# --- Detectar entorno ---
# En Local: sube niveles desde la carpeta actual hasta encontrar 'src/'
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
# RUTA_INTERIM: donde están los parquets limpios (data/01_interim/)
from src.config import RUTA_INTERIM, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    generar_tabla_con_tooltip,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Crear carpetas de salida ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
RUTA_REPORTES_CLEAN = RUTA_FASE1 / 'reportes_clean'
crear_directorios([RUTA_FASE1, RUTA_REPORTES_CLEAN])

# Atajo para formatear números en español (1.234.567)
fmt = formato_numero_es

info_entorno()

✅ Sweetviz 2.3.1
✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS LIMPIOS (desde parquets)
# ============================================================================
#
# Lee dinámicamente TODOS los parquets que hay en data/01_interim/.
# No necesita saber los nombres de antemano: busca todos los .parquet
# y los carga en un diccionario.
#
# Ejemplo: si hay 9 parquets, dfs_clean tendrá 9 entradas:
#   dfs_clean = {'becas': df, 'demograficos': df, 'expedientes': df, ...}
#
# Estos son los datos YA limpios por M02 (snake_case, NaN, tipos).
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS LIMPIOS')
print('=' * 60)

# Diccionario donde guardamos cada tabla limpia
dfs_clean = {}

# Buscar todos los parquets en la carpeta interim
for ruta in sorted(RUTA_INTERIM.glob('*.parquet')):
    nombre = ruta.stem  # nombre del archivo sin extensión
    df = pd.read_parquet(ruta)
    dfs_clean[nombre] = df
    print(f'   ✅ {nombre:15s}: {len(df):>8,} filas × {len(df.columns)} cols')

print(f'\n✅ Total: {len(dfs_clean)} tablas cargadas desde {RUTA_INTERIM}')

CARGANDO DATOS LIMPIOS
   ✅ becas          :   70,524 filas × 4 cols
   ✅ demograficos   :   30,873 filas × 5 cols
   ✅ domicilios     :  210,911 filas × 6 cols
   ✅ expedientes    :  109,568 filas × 15 cols
   ✅ notas          :  107,908 filas × 5 cols
   ✅ preinscripcion :  210,986 filas × 24 cols
   ✅ recibos        :  114,454 filas × 5 cols
   ✅ titulaciones   :       45 filas × 6 cols
   ✅ trabajo        :  195,524 filas × 4 cols

✅ Total: 9 tablas cargadas desde C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim


In [3]:
# ============================================================================
# CELDA 3: GENERAR REPORTES SWEETVIZ
# ============================================================================
#
# Para cada tabla limpia, genera un reporte Sweetviz (HTML independiente).
# Es igual que M01, pero con los datos DESPUÉS de limpiar.
#
# También calculamos métricas de calidad (filas, columnas, nulos, duplicados)
# para los KPIs y la tabla resumen del HTML.
#
# ⏱️ Esta celda tarda unos minutos (genera 9 reportes).
# ============================================================================

print('=' * 60)
print('GENERANDO REPORTES SWEETVIZ')
print('=' * 60)

# Lista donde guardamos las métricas de cada tabla
metricas_tablas = []

for nombre, df in dfs_clean.items():
    print(f'\n📊 Generando reporte: {nombre}...')

    # --- Calcular métricas de calidad ---
    n_filas = len(df)
    n_cols = len(df.columns)
    n_nulos = df.isna().sum().sum()
    total_celdas = n_filas * n_cols
    pct_nulos = (n_nulos / total_celdas * 100) if total_celdas > 0 else 0
    n_duplicados = df.duplicated().sum()

    metricas_tablas.append({
        'nombre': nombre.capitalize(),
        'filas': n_filas,
        'columnas': n_cols,
        'nulos_pct': f'{pct_nulos:.2f}%' if pct_nulos > 0 else '0.0%',
        'duplicados': n_duplicados,
        'cols_lista': list(df.columns),  # para tooltips en la tabla HTML
        'nulos_total': n_nulos
    })

    # --- Preparar datos para Sweetviz ---
    # Convertir columnas 'object' a string (Sweetviz falla con tipos mixtos)
    df_sv = df.copy()
    for col in df_sv.columns:
        if df_sv[col].dtype == 'object':
            df_sv[col] = df_sv[col].astype(str)

    # --- Generar y guardar reporte ---
    ruta_reporte = RUTA_REPORTES_CLEAN / f'reporte_{nombre}.html'
    try:
        report = sv.analyze(df_sv, pairwise_analysis='off')
        report.show_html(filepath=str(ruta_reporte), open_browser=False)
        print(f'   ✅ {ruta_reporte.name}')
    except Exception as e:
        print(f'   ❌ Error: {e}')

print(f'\n✅ {len(metricas_tablas)} reportes generados en {RUTA_REPORTES_CLEAN}')

GENERANDO REPORTES SWEETVIZ

📊 Generando reporte: becas...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean\reporte_becas.html was generated.
   ✅ reporte_becas.html

📊 Generando reporte: demograficos...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean\reporte_demograficos.html was generated.
   ✅ reporte_demograficos.html

📊 Generando reporte: domicilios...


...

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean\reporte_recibos.html was generated.
   ✅ reporte_recibos.html

📊 Generando reporte: titulaciones...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean\reporte_titulaciones.html was generated.
   ✅ reporte_titulaciones.html

📊 Generando reporte: trabajo...


                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean\reporte_trabajo.html was generated.
   ✅ reporte_trabajo.html

✅ 9 reportes generados en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean


In [4]:
# ============================================================================
# CELDA 4: CALCULAR KPIs
# ============================================================================
#
# Calcula los indicadores clave que se mostrarán en la página HTML.
# La diferencia con M01 es que aquí los nulos son "estandarizados":
# en M01 muchos valores vacíos aparecían como "-" o "N/A" (0 nulos),
# pero tras la limpieza (M02) se convirtieron a NaN real.
# Por eso el número de nulos puede SUBIR — es correcto y esperado.
# ============================================================================

total_filas = sum(m['filas'] for m in metricas_tablas)
total_cols = sum(m['columnas'] for m in metricas_tablas)
total_nulos_clean = sum(m['nulos_total'] for m in metricas_tablas)

KPIS = [
    {'valor': str(len(dfs_clean)),    'titulo': 'Tablas',               'color': '#3182ce'},
    {'valor': fmt(total_filas),       'titulo': 'Registros',            'color': '#38a169'},
    {'valor': str(total_cols),        'titulo': 'Columnas',             'color': '#ed8936'},
    {'valor': fmt(total_nulos_clean), 'titulo': 'Nulos estandarizados', 'color': '#805ad5'},
]

print('KPIs calculados:')
for kpi in KPIS:
    print(f"  {kpi['titulo']:22s}: {kpi['valor']}")

KPIs calculados:
  Tablas                : 9
  Registros             : 1.050.793
  Columnas              : 74
  Nulos estandarizados  : 364.914


In [5]:
# ============================================================================
# CELDA 5: PREPARAR TARJETAS DE REPORTES
# ============================================================================
#
# Cada tarjeta es un recuadro clicable que enlaza a un reporte Sweetviz.
# Los colores se asignan rotativamente de una paleta predefinida.
# ============================================================================

# Paleta de colores para las tarjetas
PALETA_TARJETAS = ['#3182ce', '#805ad5', '#e53e3e', '#38a169', '#ed8936', '#319795']

TARJETAS_REPORTES = []
for i, m in enumerate(metricas_tablas):
    color = PALETA_TARJETAS[i % len(PALETA_TARJETAS)]  # color rotativo
    TARJETAS_REPORTES.append({
        'titulo': m['nombre'],
        'descripcion': f"{fmt(m['filas'])} filas | {m['columnas']} cols | {m['nulos_pct']} nulos",
        'emoji': '✨',
        'link': f"reportes_clean/reporte_{m['nombre'].lower()}.html",
        'link_texto': 'Ver Sweetviz →',
        'color': color
    })

print(f'✅ {len(TARJETAS_REPORTES)} tarjetas preparadas')

✅ 9 tarjetas preparadas


In [6]:
# ============================================================================
# CELDA 6: GENERAR PÁGINA HTML
# ============================================================================
#
# Construye la página m03_reportes_clean.html con:
#   - Navegación por fases y módulos
#   - KPIs (números grandes)
#   - Tarjetas clicables enlazando a cada reporte Sweetviz
#   - Tabla resumen con métricas de calidad
#
# render_base_html() usa la plantilla base.html (Jinja2) que incluye:
#   cabecera con logos, footer con autora, enlace a GitHub.
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Navegación dinámica ---
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase1',
    modulo_activo='m03'
)

# --- KPIs ---
kpis_html = generar_kpis_html(KPIS)

# --- Sección: Tarjetas de reportes ---
tarjetas_html = generar_tarjetas_html(TARJETAS_REPORTES)
seccion_reportes = generar_seccion_html(
    titulo='Reportes Sweetviz — Datos DESPUÉS de limpiar',
    contenido=tarjetas_html,
    icono='📊'
)

# --- Sección: Tabla resumen de calidad ---
tabla_html = generar_tabla_con_tooltip(metricas_tablas)
seccion_tabla = generar_seccion_html(
    titulo='Resumen de Calidad',
    contenido=tabla_html,
    icono='📋'
)

# --- Juntar contenido ---
contenido_html = kpis_html + seccion_reportes + seccion_tabla

# --- Generar página completa con plantilla base ---
html_completo = render_base_html(
    titulo='M03: Reportes Clean',
    icono='✨',
    subtitulo='Fase 1: Transformación | TFM Abandono Universitario',
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f1_m03_reportes_clean.ipynb',
    notebook_carpeta='fase1_transformacion'
)

# --- Guardar ---
ruta_html = RUTA_FASE1 / 'm03_reportes_clean.html'
guardar_html(html_completo, ruta_html)

print(f'\n✅ HTML generado: {ruta_html}')

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m03_reportes_clean.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m03_reportes_clean.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F1-M03 COMPLETADO')
print('=' * 60)
print(f'\n📁 Archivos generados:')
print(f'   📄 {ruta_html}')
print(f'   📊 {len(metricas_tablas)} reportes Sweetviz en {RUTA_REPORTES_CLEAN}')
print(f'\n📊 KPIs: {len(dfs_clean)} tablas, {fmt(total_filas)} registros, {fmt(total_nulos_clean)} nulos')
print(f'\n📌 Siguiente: f1_m04a_union_tablas.ipynb')
print('=' * 60)


✅ F1-M03 COMPLETADO

📁 Archivos generados:
   📄 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m03_reportes_clean.html
   📊 9 reportes Sweetviz en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\reportes_clean

📊 KPIs: 9 tablas, 1.050.793 registros, 364.914 nulos

📌 Siguiente: f1_m04a_union_tablas.ipynb
